# AUTOMATIC1111 Kaggle 학생용 실행

이 Notebook에서 직접 작성한 코드는 MIT License로 공개합니다. AUTOMATIC1111, 확장 프로그램 및 모델은 각각의 라이선스를 따릅니다.

## 학생별 실행 원칙

- 학생마다 **자신의 Kaggle 계정**에서 이 노트북을 복사하여 실행합니다.
- Kaggle 계정·비밀번호를 다른 학생과 공유하지 않습니다.
- 각 학생에게 별도의 Kaggle 세션, WebUI 주소, 임시 비밀번호가 생성됩니다.
- 수업이 끝나면 생성 이미지를 내려받고 WebUI와 Kaggle 세션을 종료합니다.

## 실행 순서

1. Kaggle에 로그인한 뒤 배포받은 노트북에서 **Copy & Edit**를 누릅니다.
2. 오른쪽 **Settings**에서 **Accelerator → GPU**를 선택하고 **Internet → On**으로 설정합니다.
3. 상단의 **Run All**을 누릅니다.
4. 실행 패널 맨 위에서 사용할 **Google Drive 모델 폴더 링크**를 지정하고 **폴더 지정·목록 불러오기**를 누릅니다. Drive를 사용하지 않으면 이 단계는 건너뜁니다.
5. 사용할 Checkpoint를 하나 이상 선택합니다.
   - CivitAI·Hugging Face·Google Drive 개별 모델은 **파일 링크**에서 등록합니다.
6. 모델 이름 검색과 **이전·다음**으로 필요한 LoRA·ControlNet·Embedding·VAE를 선택합니다. 한 페이지에 25개씩 보이며, 페이지를 바꾸어도 선택은 유지됩니다.
7. **선택 모델 적용·WebUI 실행**을 누릅니다.
8. 상태가 `5/5 · 접속 준비 완료`로 바뀌면 **WebUI 열기**를 누릅니다.
9. 사용자명은 `a1111`을 입력하고 **비밀번호 복사**를 눌러 비밀번호를 붙여 넣습니다.
10. 실습을 마치면 **이미지 ZIP**으로 결과를 내려받고 **WebUI 종료**를 누릅니다.
11. Kaggle 오른쪽 위 세션 메뉴에서 **Stop Session**을 선택합니다.

## Google Drive 공유 폴더

모델 폴더를 Google Drive에서 **링크가 있는 모든 사용자 → 뷰어**로 공유합니다. 폴더 URL을 실행 패널 맨 위 **Drive 모델 폴더**에 붙여 넣고 **폴더 지정·목록 불러오기**를 누르십시오. 목록을 읽을 때는 모델 파일을 내려받지 않으며, 선택한 모델만 Kaggle에 저장됩니다.

## 모델 버튼

| 버튼 | 기능 |
|---|---|
| 사용 | 다음 WebUI 실행에 연결할 모델 선택 |
| 다운로드 | 해당 모델만 자신의 Kaggle 작업 공간에 다운로드 |
| 보관 고정 | 자동 캐시 정리에서 제외 |
| 삭제 | 자신의 Kaggle 작업 공간에 받은 복사본 삭제 |
| 작업 중단 | 진행 중인 설치 또는 다운로드 중단 |


In [ ]:
# 아래 실행 패널에서 모델을 선택한 뒤 WebUI를 실행하십시오.
import sys, subprocess, importlib.util, importlib.metadata
needed=[name for name in ('ipywidgets','requests') if importlib.util.find_spec(name) is None]
try:
    if importlib.metadata.version('gdown')!='6.4.1':needed.append('gdown==6.4.1')
except importlib.metadata.PackageNotFoundError:
    needed.append('gdown==6.4.1')
if needed:subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
import ipywidgets as W
from IPython.display import display, FileLink
import requests,gdown
import html,contextlib,io,socket,signal,tempfile,zipfile


import os, re, json, time, hashlib, shutil, threading
from pathlib import Path
from urllib.parse import urlparse, parse_qs, unquote

MODEL_SUFFIXES={'.safetensors','.ckpt','.pt','.pth','.bin'}
GIB=1024**3

class Cancelled(RuntimeError):
    pass


def model_key(item):
    parsed=urlparse(item['url'])
    identity=item['url']
    if parsed.hostname=='drive.google.com':
        match=re.search(r'/file/d/([\w-]+)',parsed.path)
        file_id=match.group(1) if match else (parse_qs(parsed.query).get('id') or [''])[0]
        if file_id:identity='drive:'+file_id
    return hashlib.sha256((item['category']+'\0'+identity).encode()).hexdigest()[:24]


def model_filename(value):
    value=unquote(str(value)).replace('\\','/').rsplit('/',1)[-1].strip()
    value=re.sub(r'[\x00-\x1f<>:"|?*]','_',value)
    if value in {'','.', '..'} or Path(value).suffix.lower() not in MODEL_SUFFIXES:
        raise ValueError('지원하는 모델 파일명(.safetensors/.ckpt/.pt/.pth/.bin)이 아닙니다.')
    suffix=Path(value).suffix
    return Path(value).stem[:150]+suffix


class ModelCache:
    def __init__(self,root,limit=20*GIB,reserve=3*GIB,cancel=None,log=None,disk_usage=None):
        self.root=Path(root).absolute()
        if self.root.is_symlink():raise RuntimeError('캐시 폴더는 심볼릭 링크일 수 없습니다.')
        self.root.mkdir(parents=True,exist_ok=True)
        self.blobs=self.root/'blobs'
        if self.blobs.is_symlink():raise RuntimeError('캐시 저장 폴더가 올바르지 않습니다.')
        self.blobs.mkdir(exist_ok=True)
        self.manifest=self.root/'models.json'
        self.limit=int(limit);self.reserve=int(reserve)
        self.cancel=cancel or threading.Event();self.log=log or (lambda msg:None)
        self.disk_usage=disk_usage or shutil.disk_usage
        self.data={'version':1,'catalog':{},'records':{},'active':[],'pins':[],'links':[]}
        if self.manifest.exists():
            obj=json.loads(self.manifest.read_text())
            if obj.get('version')!=1:raise RuntimeError('모델 관리 파일의 버전을 확인하십시오.')
            self.data.update(obj)
        self.protected=set(self.data['active'])
        self.lock=threading.RLock()

    def check(self):
        if self.cancel.is_set():raise Cancelled('작업을 중단했습니다.')

    def save(self):
        with self.lock:
            tmp=self.manifest.with_suffix('.tmp')
            tmp.write_text(json.dumps(self.data,ensure_ascii=False,indent=2))
            tmp.replace(self.manifest)

    def register(self,item):
        item=dict(item)
        key=model_key(item)
        self.data['catalog'][key]=item
        self.save()
        return key

    def path_for(self,key,name):
        if not re.fullmatch('[a-f0-9]{24}',key):raise ValueError('올바르지 않은 모델 식별자입니다.')
        folder=self.blobs/key
        if folder.is_symlink():raise RuntimeError('캐시 내부 링크는 사용할 수 없습니다.')
        path=folder/model_filename(name)
        if path.is_symlink():raise RuntimeError('캐시 파일이 링크로 바뀌어 있습니다.')
        if not path.resolve().is_relative_to(self.blobs.resolve()):raise RuntimeError('허용되지 않은 파일 경로입니다.')
        return path

    def cached_path(self,key):
        record=self.data['records'].get(key)
        if not record:return None
        path=self.path_for(key,record['filename'])
        if path.is_file() and path.stat().st_size==record['size'] and record['size']>0:return path
        return None

    def usage(self):
        total=0
        for path in self.blobs.glob('*/*'):
            if path.is_file() and not path.is_symlink() and not path.parent.is_symlink():total+=path.stat().st_size
        return total

    def pin(self,key,value):
        pins=set(self.data['pins'])
        if value:pins.add(key)
        else:pins.discard(key)
        self.data['pins']=sorted(pins);self.save()

    def delete(self,key,manual=False):
        if key in self.data['active'] or key in self.protected:
            raise RuntimeError('현재 사용 중이거나 이번 작업에서 필요한 모델은 삭제할 수 없습니다.')
        if key in self.data['pins']:
            raise RuntimeError('보관 고정을 먼저 해제하십시오.')
        record=self.data['records'].get(key)
        if not record:return False
        path=self.path_for(key,record['filename'])
        # 모델 관리 기능으로 받은 파일만 삭제합니다.
        if path.exists():path.unlink()
        try:path.parent.rmdir()
        except OSError:pass
        self.data['records'].pop(key,None);self.save()
        self.log(('삭제: ' if manual else '자동 정리: ')+record['filename'])
        return True

    def ensure_room(self,additional=0):
        self.check()
        while self.usage()+additional>self.limit or self.disk_usage(self.root).free-additional<self.reserve:
            candidates=[(r.get('last_used',0),key) for key,r in self.data['records'].items()
                        if key not in self.protected and key not in self.data['active'] and key not in self.data['pins']]
            if not candidates:
                raise RuntimeError('선택·사용·보관 고정 모델을 유지할 공간이 부족합니다. 선택 모델을 줄이거나 보관 고정을 해제하거나 모델 한도를 조정하십시오. 설치 파일·기존 파일이 디스크를 차지할 수도 있습니다.')
            _,key=min(candidates)
            self.delete(key)

    def clean_unselected(self):
        for key in list(self.data['records']):
            if key not in self.protected and key not in self.data['active'] and key not in self.data['pins']:
                self.delete(key)

    def fetch(self,item,resolve,request_headers,session_get,drive_download):
        self.check()
        key=self.register(item)
        self.protected.add(key)
        cached=self.cached_path(key)
        if cached:
            self.data['records'][key]['last_used']=time.time();self.save()
            self.log('재사용: '+cached.name)
            return cached
        old_record=self.data['records'].get(key)
        if old_record:
            if key in self.data['active']:
                raise RuntimeError('사용 중인 모델의 파일 상태가 바뀌었습니다. WebUI를 종료한 뒤 다시 다운로드하십시오.')
            old_path=self.path_for(key,old_record['filename'])
            old_path.unlink(missing_ok=True)
            self.data['records'].pop(key,None);self.save()
        url=item['url'];name=item.get('filename','')
        if urlparse(url).hostname in {'civitai.com','www.civitai.com'}:
            url,fallback=resolve(url)
            name=name or fallback
        if not name:name=unquote(Path(urlparse(url).path).name)
        name=model_filename(name)
        if item['category']=='Checkpoint' and Path(name).suffix.lower() not in {'.safetensors','.ckpt'}:
            raise ValueError('Checkpoint는 .safetensors 또는 .ckpt 파일을 사용하십시오.')
        target=self.path_for(key,name);target.parent.mkdir(exist_ok=True)
        partial=target.with_name(target.name+'.part')
        if partial.is_symlink():raise RuntimeError('임시 다운로드 경로가 링크입니다.')
        partial.unlink(missing_ok=True)
        self.ensure_room(0)
        started=time.monotonic();last=[0.0]
        total_hint=[None]
        cache=self
        class BoundedWriter:
            def __init__(self,handle):self.handle=handle;self.done=0
            def write(self,chunk):
                cache.check();cache.ensure_room(len(chunk))
                count=self.handle.write(chunk)
                self.handle.flush()
                self.done+=count
                now=time.monotonic()
                if now-last[0]>3:
                    suffix=f' / {total_hint[0]/GIB:.2f} GiB' if total_hint[0] else ' GiB'
                    cache.log(f'{name}: {self.done/GIB:.2f}'+suffix)
                    last[0]=now
                return count
            def flush(self):self.handle.flush()
            def tell(self):return self.handle.tell()
        def progress(done,total):
            cache.check()
            if total and total_hint[0] is None:
                total_hint[0]=total
                cache.ensure_room(max(0,total-done))
        try:
            self.log('다운로드: '+name)
            with partial.open('wb',buffering=0) as handle:
                writer=BoundedWriter(handle)
                if urlparse(url).hostname=='drive.google.com':
                    result=drive_download(url=url,output=writer,quiet=True,use_cookies=False,
                                          timeout=(15,30),cancel=self.cancel,progress=progress)
                    if result is None:raise RuntimeError('Google Drive 다운로드에 실패했습니다.')
                else:
                    headers=request_headers(url);headers['Accept-Encoding']='identity'
                    with session_get(url,headers=headers,stream=True,timeout=(15,30)) as response:
                        if not response.ok:raise RuntimeError(f'모델 다운로드 실패: HTTP {response.status_code}')
                        if any(t in response.headers.get('content-type','').lower() for t in ('text/html','application/json')):
                            raise RuntimeError('모델 대신 웹페이지를 받았습니다. 공유 권한·링크·토큰을 확인하십시오.')
                        if not item.get('filename'):
                            disposition=response.headers.get('content-disposition','')
                            match=re.search(r"filename\*=UTF-8''([^;]+)",disposition,re.I) or re.search(r'filename="?([^";]+)',disposition,re.I)
                            if match:
                                name=model_filename(match.group(1))
                                target=self.path_for(key,name)
                                if item['category']=='Checkpoint' and target.suffix.lower() not in {'.safetensors','.ckpt'}:
                                    raise ValueError('선택한 파일은 지원되는 Checkpoint 형식이 아닙니다.')
                        total_hint[0]=int(response.headers.get('content-length') or 0) or None
                        self.ensure_room(total_hint[0] or 0)
                        for chunk in response.iter_content(4*1024*1024):
                            if chunk:writer.write(chunk)
            self.check()
            size=partial.stat().st_size
            if not size or (total_hint[0] and size!=total_hint[0]):raise RuntimeError('모델 파일이 완전히 다운로드되지 않았습니다.')
            with partial.open('rb') as handle:prefix=handle.read(512).lstrip().lower()
            if prefix.startswith((b'<html',b'<!doctype html',b'version https://git-lfs')):
                raise RuntimeError('모델 대신 HTML 또는 Git LFS 포인터를 받았습니다.')
            partial.replace(target)
            self.data['records'][key]={'filename':name,'size':size,'last_used':time.time()}
            self.save();self.log(f'완료: {name} ({size/GIB:.2f} GiB)')
            return target
        except BaseException:
            partial.unlink(missing_ok=True)
            try:target.parent.rmdir()
            except OSError:pass
            raise

    def unlink_managed(self,webui):
        roots=model_directories(webui)
        for folder in roots.values():
            if not folder.exists():continue
            for link in folder.glob('km_*'):
                if re.match(r'^km_[a-f0-9]{24}__',link.name) and link.is_symlink():
                    target=Path(os.readlink(link))
                    if not target.is_absolute():target=link.parent/target
                    if target.resolve().is_relative_to(self.blobs.resolve()):link.unlink()
        self.data['links']=[];self.data['active']=[];self.save()

    def activate(self,items,webui):
        paths={}
        for item in items:
            key=model_key(item)
            path=self.cached_path(key)
            if path is None:raise RuntimeError('선택한 모델이 아직 다운로드되지 않았습니다.')
            paths[key]=path
        self.unlink_managed(webui)
        roots=model_directories(webui);links=[];first=None
        for item in items:
            key=model_key(item);path=paths[key];folder=roots[item['category']]
            folder.mkdir(parents=True,exist_ok=True)
            link=folder/f'km_{key}__{path.name}'
            if link.exists() or link.is_symlink():raise RuntimeError('이미 존재하는 파일을 덮어쓸 수 없습니다: '+link.name)
            link.symlink_to(path)
            links.append(str(link))
            if item['category']=='Checkpoint' and first is None:first=link
        self.data['active']=list(paths);self.data['links']=links;self.save()
        return first


def model_directories(webui):
    webui=Path(webui)
    return {'Checkpoint':webui/'models/Stable-diffusion','LoRA':webui/'models/Lora',
            'Embedding':webui/'embeddings','VAE':webui/'models/VAE',
            'ControlNet':webui/'extensions/sd-webui-controlnet/models',
            'Motion':webui/'extensions/sd-webui-animatediff/model'}


STAGES = {'환경 확인': "# 3. Kaggle 환경 점검 및 폴더 준비\nimport os\nimport sys\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nROOT = Path('/kaggle/working')\nif not ROOT.is_dir():\n    raise RuntimeError('Kaggle Notebook에서 실행하십시오.')\nif not shutil.which('nvidia-smi'):\n    raise RuntimeError('Settings → Accelerator에서 GPU를 선택하고 세션을 다시 시작하십시오.')\n\ngpu_check = subprocess.run(\n    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],\n    capture_output=True, text=True, timeout=30,\n)\nif gpu_check.returncode != 0 or not gpu_check.stdout.strip():\n    raise RuntimeError('GPU를 사용할 수 없습니다. Accelerator 설정과 GPU 할당 상태를 확인하십시오.')\ngpu_rows = gpu_check.stdout.strip().splitlines()\nif not isinstance(GPU_INDEX, int) or GPU_INDEX < 0 or GPU_INDEX >= len(gpu_rows):\n    raise ValueError('GPU_INDEX가 할당된 GPU 범위를 벗어났습니다.')\nprint('노트북 Python:', sys.version.split()[0])\nprint('할당된 GPU:\\n' + gpu_check.stdout)\nprint('선택한 GPU:', GPU_INDEX)\n\nWEBUI = ROOT / 'stable-diffusion-webui'\nVENV = ROOT / 'venv-sd310'\nPYTHON = VENV / 'bin/python'\nDOWNLOAD_ROOT = ROOT / 'downloads'\nGENERATED = DOWNLOAD_ROOT / 'generated'\nGENERATED.mkdir(parents=True, exist_ok=True)\nos.chdir(ROOT)\n\nprobe_env = os.environ.copy()\nprobe_env['GIT_TERMINAL_PROMPT'] = '0'\nprobe = subprocess.run(\n    ['git', 'ls-remote', '--heads', 'https://github.com/AUTOMATIC1111/stable-diffusion-webui.git'],\n    env=probe_env, capture_output=True, text=True, timeout=60,\n)\nif probe.returncode != 0:\n    raise RuntimeError('GitHub에 연결하지 못했습니다. Internet 설정과 아래 오류를 확인하십시오.\\n' + probe.stderr[-2000:])\nprint('환경 점검 완료. 이미지 저장 경로:', GENERATED)\n", 'Python·WebUI 설치 및 GPU 확인': '# 4. Python 3.10과 AUTOMATIC1111 설치\nimport ctypes.util\nimport importlib\nimport subprocess\n\nif not ctypes.util.find_library(\'GL\') or not ctypes.util.find_library(\'glib-2.0\'):\n    run([\'apt-get\', \'update\', \'-qq\'])\n    run([\'apt-get\', \'install\', \'-y\', \'-qq\', \'libgl1\', \'libglib2.0-0\'])\nrun([sys.executable, \'-m\', \'pip\', \'install\', \'-q\', \'uv\'])\nimportlib.invalidate_caches()\nif not PYTHON.exists():\n    run([sys.executable, \'-m\', \'uv\', \'venv\', \'--python\', \'3.10\', \'--seed\', VENV])\nrun([PYTHON, \'-m\', \'pip\', \'install\', \'-q\', \'pip<24.1\', \'setuptools<70\', \'wheel\'])\n\nif not WEBUI.exists():\n    run([\'git\', \'clone\', \'--branch\', A1111_VERSION, \'--depth\', \'1\',\n         \'https://github.com/AUTOMATIC1111/stable-diffusion-webui.git\', WEBUI], env=probe_env)\nelse:\n    if not (WEBUI / \'.git\').is_dir():\n        raise RuntimeError(\'WebUI 폴더에 Git 저장소가 없습니다. 새 Kaggle 세션에서 실행하십시오.\')\n    run([\'git\', \'fetch\', \'--depth\', \'1\', \'origin\', \'tag\', A1111_VERSION], cwd=WEBUI, env=probe_env)\n    run([\'git\', \'checkout\', \'--detach\', A1111_VERSION], cwd=WEBUI, env=probe_env)\n\nlaunch_env = os.environ.copy()\nlaunch_env.update({\n    \'STABLE_DIFFUSION_REPO\': \'https://github.com/w-e-w/stablediffusion.git\',\n    \'GRADIO_ANALYTICS_ENABLED\': \'False\',\n    \'PYTHONNOUSERSITE\': \'1\',\n    \'PYTHONUNBUFFERED\': \'1\',\n    \'MPLBACKEND\': \'Agg\',\n    \'GIT_TERMINAL_PROMPT\': \'0\',\n    \'CUDA_VISIBLE_DEVICES\': str(GPU_INDEX),\n    \'PIP_NO_CACHE_DIR\': \'1\',\n    \'TORCH_COMMAND\': \'pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121\',\n})\nlaunch_env.pop(\'PYTHONPATH\', None)\nlaunch_env.pop(\'COMMANDLINE_ARGS\', None)\nrun([PYTHON, \'launch.py\', \'--exit\'], cwd=WEBUI, env=launch_env)\n\n# WebUI에서 사용하는 Python 환경으로 GPU 연산을 확인합니다.\ncuda_probe = """\nimport torch\nassert torch.cuda.is_available(), \'CUDA를 사용할 수 없습니다.\'\nx = torch.ones((256, 256), device=\'cuda\')\ny = x @ x\ntorch.cuda.synchronize()\nprint(\'WebUI PyTorch:\', torch.__version__)\nprint(\'WebUI CUDA:\', torch.version.cuda)\nprint(\'실제 연산 장치:\', y.device, \'/\', torch.cuda.get_device_name(0))\n"""\nrun([PYTHON, \'-c\', cuda_probe], env=launch_env)\nprint(\'설치 및 GPU 연산 확인 완료\')\n', '다운로드 함수 준비': 'def request_headers(url):\n    headers = HEADERS.copy()\n    if CIVITAI_TOKEN and urlparse(url).scheme == \'https\' and urlparse(url).hostname in {\'civitai.com\', \'www.civitai.com\'}:\n        headers[\'Authorization\'] = f\'Bearer {CIVITAI_TOKEN}\'\n    if HF_TOKEN and urlparse(url).scheme == \'https\' and urlparse(url).hostname == \'huggingface.co\':\n        headers[\'Authorization\'] = f\'Bearer {HF_TOKEN}\'\n    return headers\n\ndef check_status(response):\n    if not response.ok:\n        raise RuntimeError(f\'다운로드/API 요청 실패: HTTP {response.status_code}. 링크·접근 권한·토큰을 확인하십시오.\')\n\ndef safe_name(name):\n    name = unquote(name).replace(\'\\\\\', \'_\').replace(\'/\', \'_\').strip()\n    name = re.sub(r\'[^0-9A-Za-z가-힣._() +\\-]\', \'_\', name)\n    if not name or name in {\'.\', \'..\'}:\n        raise ValueError(\'올바른 모델 파일명을 입력하십시오.\')\n    return name\n\ndef filename_from_response(response, fallback):\n    cd = response.headers.get(\'content-disposition\', \'\')\n    m = re.search(r"filename\\*=UTF-8\'\'([^;]+)", cd, re.I)\n    if not m:\n        m = re.search(r\'filename="?([^";]+)\', cd, re.I)\n    if m:\n        return safe_name(m.group(1))\n    path_name = Path(urlparse(response.url).path).name\n    return safe_name(path_name if \'.\' in path_name else fallback)\n\ndef stream_download(url, dest_dir, fallback=\'model.safetensors\', explicit_name=None):\n    dest_dir.mkdir(parents=True, exist_ok=True)\n    with requests.get(url, headers=request_headers(url), stream=True, allow_redirects=True, timeout=(30, 300)) as response:\n        check_status(response)\n        content_type = response.headers.get(\'content-type\', \'\').lower()\n        if \'text/html\' in content_type or \'application/json\' in content_type:\n            raise RuntimeError(\'모델 파일 대신 웹페이지/API 메시지를 받았습니다. 다운로드 링크와 권한을 확인하십시오.\')\n        filename = safe_name(explicit_name) if explicit_name else filename_from_response(response, fallback)\n        target = dest_dir / filename\n        total = int(response.headers.get(\'content-length\', 0))\n        if target.exists() and total and target.stat().st_size == total:\n            print(\'✅ 이미 존재:\', target.name)\n            return target\n        temp = target.with_suffix(target.suffix + \'.part\')\n        with open(temp, \'wb\') as f, tqdm(total=total or None, unit=\'B\', unit_scale=True, desc=filename) as bar:\n            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):\n                if chunk:\n                    f.write(chunk)\n                    bar.update(len(chunk))\n        if not temp.stat().st_size or (total and temp.stat().st_size != total):\n            raise RuntimeError(\'파일 다운로드가 끝나지 않았습니다. 다운로드 셀을 다시 실행하십시오.\')\n        temp.replace(target)\n    print(\'✅ 다운로드:\', target)\n    return target\n\ndef civitai_file(url):\n    parsed = urlparse(url)\n    if parsed.scheme != \'https\' or parsed.hostname not in {\'civitai.com\', \'www.civitai.com\'}:\n        raise ValueError(\'CivitAI 목록에는 https://civitai.com 링크를 입력하십시오.\')\n    query = parse_qs(parsed.query)\n    version_match = re.search(r\'/api/download/models/(\\d+)\', parsed.path)\n    version_id = (query.get(\'modelVersionId\') or [None])[0]\n    if version_match:\n        return url, f\'civitai_{version_match.group(1)}.safetensors\'\n\n    if version_id:\n        api_url = f\'https://civitai.com/api/v1/model-versions/{version_id}\'\n        info = requests.get(api_url, headers=request_headers(api_url), timeout=(15,30))\n        check_status(info)\n        version = info.json()\n    else:\n        model_match = re.search(r\'/models/(\\d+)\', parsed.path)\n        if not model_match:\n            raise ValueError(\'CivitAI 모델 페이지 또는 직접 다운로드 링크를 입력하십시오.\')\n        api_url = f\'https://civitai.com/api/v1/models/{model_match.group(1)}\'\n        info = requests.get(api_url, headers=request_headers(api_url), timeout=(15,30))\n        check_status(info)\n        versions = info.json().get(\'modelVersions\') or []\n        if not versions:\n            raise RuntimeError(\'사용 가능한 CivitAI 모델 버전이 없습니다.\')\n        version = versions[0]\n\n    files = [f for f in (version.get(\'files\') or [])\n             if Path(f.get(\'name\', \'\')).suffix.lower() in {\'.safetensors\', \'.ckpt\', \'.pt\', \'.bin\'}]\n    if not files:\n        download_url = version.get(\'downloadUrl\')\n        if not download_url:\n            raise RuntimeError(\'CivitAI API 응답에 다운로드 파일이 없습니다.\')\n        return download_url, f\'civitai_{version.get("id", "model")}.safetensors\'\n\n    preferred = sorted(\n        files,\n        key=lambda f: (\n            not bool(f.get(\'primary\')),\n            str(f.get(\'metadata\', {}).get(\'format\', \'\')).lower() != \'safetensor\',\n            str(f.get(\'name\', \'\'))\n        )\n    )[0]\n    download_url = preferred.get(\'downloadUrl\') or version.get(\'downloadUrl\')\n    if not download_url:\n        raise RuntimeError(\'선택된 CivitAI 파일에 downloadUrl이 없습니다.\')\n    return download_url, preferred.get(\'name\') or f\'civitai_{version.get("id", "model")}.safetensors\'\n\ndef download_civitai(items, dest_dir, label):\n    for item in items:\n        if not str(item).strip():\n            continue\n        print(f\'\\n[{label}] CivitAI 다운로드 중\')\n        download_url, fallback = civitai_file(str(item).strip())\n        stream_download(download_url, dest_dir, fallback)\n\ndef download_gdrive(items, dest_dir, label):\n    for item in items:\n        if not item:\n            continue\n        if not isinstance(item, (tuple, list)) or len(item) != 2:\n            raise ValueError(f\'{label} Google Drive 항목은 (공유 링크, 파일명) 형식이어야 합니다: {item!r}\')\n        url, filename = item\n        filename = safe_name(filename)\n        target = dest_dir / filename\n        print(f\'\\n[{label}] Google Drive 다운로드:\', filename)\n        if target.exists() and target.stat().st_size > 0:\n            print(\'✅ 이미 존재:\', target)\n            continue\n        temp = target.with_suffix(target.suffix + \'.part\')\n        result = gdown.download(url=str(url), output=str(temp), quiet=False, fuzzy=True, resume=True)\n        if not result or not temp.exists() or not temp.stat().st_size:\n            raise RuntimeError(f\'Google Drive 다운로드 실패: {filename}. 공유 권한과 링크를 확인하십시오.\')\n        with temp.open(\'rb\') as model_file:\n            prefix = model_file.read(512).lstrip().lower()\n        if prefix.startswith((b\'<!doctype html\', b\'<html\')):\n            temp.unlink()\n            raise RuntimeError(\'Google Drive에서 모델 대신 웹페이지를 받았습니다. 공유 권한을 확인하십시오.\')\n        temp.replace(target)', 'WebUI 시작 및 접속 준비': "# 7. 이미지 저장 경로 설정 및 WebUI 실행\n# 이 셀에 표시되는 접속 주소와 로그인 정보를 사용하십시오.\nimport json\nimport re\nimport time\nimport secrets\nimport tempfile\nimport requests\n\nfor process_name in ('tunnel_process', 'webui_process'):\n    process = globals().get(process_name)\n    if process is not None and process.poll() is None:\n        process.terminate()\n        try:\n            process.wait(timeout=15)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait(timeout=10)\n\nCONFIG = WEBUI / 'config.json'\nconfig = json.loads(CONFIG.read_text()) if CONFIG.exists() else {}\nconfig.update({\n    'samples_save': True,\n    'grid_save': True,\n    'outdir_samples': '',\n    'outdir_txt2img_samples': str(GENERATED / 'txt2img'),\n    'outdir_img2img_samples': str(GENERATED / 'img2img'),\n    'outdir_extras_samples': str(GENERATED / 'extras'),\n    'outdir_grids': '',\n    'outdir_txt2img_grids': str(GENERATED / 'grids' / 'txt2img'),\n    'outdir_img2img_grids': str(GENERATED / 'grids' / 'img2img'),\n    'outdir_save': str(GENERATED / 'saved'),\n    'sd_model_checkpoint': str(START_CHECKPOINT),\n})\nCONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2))\n\nCLOUDFLARED = ROOT / 'cloudflared'\nif not CLOUDFLARED.exists() or not CLOUDFLARED.stat().st_size:\n    cloudflare_url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'\n    temp_binary = CLOUDFLARED.with_suffix('.part')\n    with requests.get(cloudflare_url, stream=True, timeout=(15, 30)) as response:\n        response.raise_for_status()\n        with temp_binary.open('wb') as output:\n            for chunk in response.iter_content(1024 * 1024):\n                check_cancel()\n                if chunk:\n                    output.write(chunk)\n    temp_binary.replace(CLOUDFLARED)\nCLOUDFLARED.chmod(0o755)\n\nWEBUI_USER = 'a1111'\nWEBUI_PASSWORD = secrets.token_urlsafe(18)\nold_auth = globals().get('AUTH_FILE')\nif old_auth:\n    Path(old_auth).unlink(missing_ok=True)\nauth_fd, auth_name = tempfile.mkstemp(prefix='a1111-auth-', suffix='.txt')\nAUTH_FILE = Path(auth_name)\nwith os.fdopen(auth_fd, 'w') as auth:\n    auth.write(f'{WEBUI_USER}:{WEBUI_PASSWORD}')\n\nwebui_log = ROOT / 'webui.log'\nargs = [str(PYTHON), 'launch.py', '--port', '7860',\n        '--ui-settings-file', str(CONFIG), '--ckpt', str(START_CHECKPOINT), '--gradio-auth-path', str(AUTH_FILE), '--lora-dir', str(ACTIVE_LORA_DIR),\n        '--opt-sdp-attention', '--no-half-vae', '--disable-console-progressbars']\nif USE_MEDVRAM:\n    args.append('--medvram')\nwith webui_log.open('w') as log:\n    webui_process = subprocess.Popen(args, cwd=WEBUI, env=launch_env,\n                                     stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n\nTUNNEL_LOG = ROOT / 'cloudflared.log'\npublic_url = None\ntry:\n    for attempt in range(300):\n        check_cancel()\n        if webui_process.poll() is not None:\n            print(webui_log.read_text(errors='ignore')[-10000:])\n            raise RuntimeError('WebUI가 종료되었습니다. 위 실행 로그를 확인하십시오.')\n        try:\n            local_response = requests.get('http://127.0.0.1:7860/', timeout=2)\n            if local_response.status_code in (200, 401):\n                break\n        except requests.RequestException:\n            pass\n        if attempt % 15 == 0:\n            print('WebUI 준비 중… 로그 확인: /kaggle/working/webui.log', flush=True)\n        time.sleep(2)\n    else:\n        print(webui_log.read_text(errors='ignore')[-10000:])\n        raise TimeoutError('WebUI 준비 시간이 초과되었습니다. 로그를 확인하고 이 셀을 다시 실행하십시오.')\n\n    with TUNNEL_LOG.open('w') as log:\n        tunnel_process = subprocess.Popen(\n            [str(CLOUDFLARED), 'tunnel', '--url', 'http://127.0.0.1:7860',\n             '--protocol', 'http2', '--no-autoupdate'],\n            stdout=log, stderr=subprocess.STDOUT, start_new_session=True,\n        )\n    for attempt in range(60):\n        check_cancel()\n        if tunnel_process.poll() is not None:\n            raise RuntimeError('Cloudflare 터널이 종료되었습니다. cloudflared.log를 확인하십시오.')\n        tunnel_text = TUNNEL_LOG.read_text(errors='ignore')\n        match = re.search(r'https://[-\\w]+\\.trycloudflare\\.com', tunnel_text)\n        if match:\n            public_url = match.group(0)\n        if public_url and 'Registered tunnel connection' in tunnel_text:\n            break\n        time.sleep(2)\n    else:\n        print(TUNNEL_LOG.read_text(errors='ignore')[-4000:])\n        raise TimeoutError('터널 연결이 완료되지 않았습니다. 이 셀을 다시 실행하십시오.')\n    if webui_process.poll() is not None:\n        raise RuntimeError('WebUI가 종료되었습니다. webui.log를 확인하십시오.')\nexcept BaseException:\n    for process_name in ('tunnel_process', 'webui_process'):\n        process = globals().get(process_name)\n        if process is not None and process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait(timeout=10)\n    AUTH_FILE.unlink(missing_ok=True)\n    raise\n\n"}

def preset(label, category, family, repo, filename, save_as=None):
    return dict(label=label, category=category, family=family,
                url=f'https://huggingface.co/{repo}/resolve/main/{filename}',
                filename=save_as or filename.rsplit('/', 1)[-1])

PRESETS = [
    preset('DreamShaper 8', 'Checkpoint', 'SD 1.5', 'Lykon/DreamShaper', 'DreamShaper_8_pruned.safetensors'),
    preset('CyberRealistic 4.1', 'Checkpoint', 'SD 1.5', 'cyberdelia/CyberRealistic', 'CyberRealistic_V4.1_FP16.safetensors'),
    preset('Realistic Vision 6.0 B1 (별도 VAE 선택 권장)', 'Checkpoint', 'SD 1.5', 'SG161222/Realistic_Vision_V6.0_B1_noVAE', 'Realistic_Vision_V6.0_NV_B1_fp16.safetensors'),
    preset('SDXL 1.0 Base', 'Checkpoint', 'SDXL', 'stabilityai/stable-diffusion-xl-base-1.0', 'sd_xl_base_1.0.safetensors'),
    preset('SDXL 1.0 Refiner (Base와 함께 선택)', 'Checkpoint', 'SDXL', 'stabilityai/stable-diffusion-xl-refiner-1.0', 'sd_xl_refiner_1.0.safetensors'),
    preset('Juggernaut XL v9', 'Checkpoint', 'SDXL', 'RunDiffusion/Juggernaut-XL-v9', 'Juggernaut-XL_v9_RunDiffusionPhoto_v2.safetensors'),
    preset('Detail Tweaker', 'LoRA', 'SD 1.5', 'OedoSoldier/detail-tweaker-lora', 'add_detail.safetensors'),
    preset('LCM LoRA (WebUI에서 LCM 샘플러 선택)', 'LoRA', 'SD 1.5', 'latent-consistency/lcm-lora-sdv1-5', 'pytorch_lora_weights.safetensors', 'lcm-lora-sdv1-5.safetensors'),
    preset('SD 1.5 VAE · ft-mse', 'VAE', 'SD 1.5', 'stabilityai/sd-vae-ft-mse-original', 'vae-ft-mse-840000-ema-pruned.safetensors'),
]
for label, filename in [
    ('Canny · 윤곽선', 'control_v11p_sd15_canny.pth'),
    ('Depth · 깊이', 'control_v11f1p_sd15_depth.pth'),
    ('OpenPose · 자세', 'control_v11p_sd15_openpose.pth'),
    ('Lineart · 선화', 'control_v11p_sd15_lineart.pth'),
    ('SoftEdge · 부드러운 경계', 'control_v11p_sd15_softedge.pth'),
    ('Scribble · 낙서', 'control_v11p_sd15_scribble.pth'),
    ('Tile · 세부 묘사', 'control_v11f1e_sd15_tile.pth'),
    ('Inpaint · 부분 수정', 'control_v11p_sd15_inpaint.pth'),
]:
    PRESETS.append(preset(label, 'ControlNet', 'SD 1.5', 'lllyasviel/ControlNet-v1-1', filename))
for label, filename in [
    ('Canny · 윤곽선', 'diffusers_xl_canny_full.safetensors'),
    ('Depth · 깊이', 'diffusers_xl_depth_full.safetensors'),
    ('OpenPose · 자세', 'thibaud_xl_openpose.safetensors'),
]:
    PRESETS.append(preset(label, 'ControlNet', 'SDXL', 'lllyasviel/sd_control_collection', filename))
EXTENSIONS = {
    'ControlNet': 'https://github.com/Mikubill/sd-webui-controlnet.git',
    'ADetailer': 'https://github.com/Bing-su/adetailer.git',
    'Ultimate SD Upscale': 'https://github.com/Coyote-A/ultimate-upscale-for-automatic1111.git',
    'Regional Prompter': 'https://github.com/hako-mikan/sd-webui-regional-prompter.git',
    'Dynamic Thresholding': 'https://github.com/mcmonkeyprojects/sd-dynamic-thresholding.git',
    'Dynamic Prompts': 'https://github.com/adieyal/sd-dynamic-prompts.git',
    'StyleSelectorXL': 'https://github.com/ahgsql/StyleSelectorXL.git',
    'Inpaint Anything': 'https://github.com/Uminosachi/sd-webui-inpaint-anything.git',
    'AnimateDiff': 'https://github.com/continue-revolution/sd-webui-animatediff.git',
    'SadTalker': 'https://github.com/OpenTalker/SadTalker.git',
}



def connection_html(url, username, password):
    u=html.escape(url,quote=True)
    user=html.escape(username,quote=True)
    pw=html.escape(password,quote=True)
    return f"""<div style="padding:22px;border:3px solid #15803d;border-radius:12px;background:#f0fdf4;color:#14352a">
    <div style="font-size:24px;font-weight:800">✓ 준비 완료 — 학생 개인 WebUI</div>
    <p><a href="{u}" target="_blank" rel="noopener noreferrer" style="display:inline-block;padding:16px 28px;background:#15803d;color:white;font-weight:800;font-size:21px;border-radius:8px;text-decoration:none">AUTOMATIC1111 WebUI 열기 ↗</a></p>
    <div style="padding:10px 12px;margin:12px 0;background:white;border:1px solid #86efac;border-radius:8px">
      <div style="margin-bottom:10px"><b>AUTOMATIC1111 사용자명:</b> <code style="font-size:16px">{user}</code> <small>(Kaggle 계정 ID가 아닙니다)</small></div>
      <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap">
        <b>임시 비밀번호:</b>
        <input id="a1111-webui-password" name="a1111-session-password" autocomplete="new-password" type="password" value="{pw}" readonly style="padding:9px;width:250px;border:1px solid #94a3b8;border-radius:6px;font-family:monospace" aria-label="AUTOMATIC1111 임시 비밀번호">
        <button onclick="const b=this,v='{pw}',done=()=>{{b.textContent='비밀번호 복사 완료 ✓';setTimeout(()=>b.textContent='비밀번호 복사',1800)}},fallback=()=>{{const t=document.createElement('textarea');t.value=v;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.focus();t.select();const ok=document.execCommand('copy');t.remove();ok?done():b.textContent='복사 실패 · 표시 후 직접 복사'}};if(navigator.clipboard&&window.isSecureContext){{navigator.clipboard.writeText(v).then(done).catch(fallback)}}else{{fallback()}}" style="padding:9px 13px;border:0;border-radius:6px;background:#2563eb;color:white;font-weight:700;cursor:pointer">비밀번호 복사</button>
        <button onclick="const p=document.getElementById('a1111-webui-password');p.type=p.type==='password'?'text':'password';this.textContent=p.type==='password'?'비밀번호 표시':'비밀번호 숨기기'" style="padding:9px 13px;border:1px solid #64748b;border-radius:6px;background:white;cursor:pointer">비밀번호 표시</button>
      </div>
    </div>
    <p><b>로그인 화면에 브라우저가 다른 값을 자동 입력하면 지우고, 사용자명 <code>{user}</code>과 위 임시 비밀번호를 입력하십시오.</b></p>
    <p><small>이 정보는 Kaggle 계정이 아니라 현재 학생 세션의 AUTOMATIC1111 임시 로그인 정보입니다.</small></p>
    <p>버튼이 열리지 않으면 이 주소를 복사해 새 탭에 붙여 넣으십시오.<br><code style="overflow-wrap:anywhere">{u}</code></p>
    <p>WebUI에서 프롬프트 입력 → <b>Generate</b>를 누르면 이미지가 생성됩니다.<br>새 탭에서 이미지를 생성하는 동안 이 Kaggle 세션을 유지하십시오.</p></div>"""

WORK_ROOT=Path('/kaggle/working')
WEBUI_ROOT=WORK_ROOT/'stable-diffusion-webui'
if not WORK_ROOT.is_dir():raise RuntimeError('Kaggle Notebook에서 이 노트북을 실행하십시오.')
OLD=globals().get('MODEL_MANAGER_SESSION')
if OLD and OLD.get('worker') and OLD['worker'].is_alive():raise RuntimeError('진행 중인 작업을 먼저 중단한 뒤 패널을 다시 실행하십시오.')
STATE=OLD or {'runtime':globals().get('A1111_RUNTIME',{}),'worker':None,'process':None,'cancel':threading.Event()}
MODEL_MANAGER_SESSION=STATE
RUNTIME=STATE['runtime']
CATALOG={}
SELECTED=set()
SECRET_VALUES=[]


def emit(message):
    text=str(message)
    for value in SECRET_VALUES:
        if value:text=text.replace(value,'[토큰 숨김]')
    logs.value=(logs.value+text+'\n')[-30000:]


def check_cancel():
    if STATE['cancel'].is_set():raise Cancelled('작업을 중단했습니다.')


def terminate(process,wait=True):
    if process is None or process.poll() is not None:return
    try:
        if os.getpgid(process.pid)==process.pid:os.killpg(process.pid,signal.SIGTERM)
        else:process.terminate()
        if wait:
            try:process.wait(timeout=4)
            except subprocess.TimeoutExpired:
                if os.getpgid(process.pid)==process.pid:os.killpg(process.pid,signal.SIGKILL)
                else:process.kill()
                process.wait(timeout=4)
    except ProcessLookupError:pass


def run(args,cwd=None,env=None):
    check_cancel()
    p=subprocess.Popen([str(x) for x in args],cwd=cwd,env=env,stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    STATE['process']=p
    try:
        for line in p.stdout:
            check_cancel();emit(line.rstrip())
        result=p.wait();check_cancel()
        if result:raise RuntimeError('명령'+' 실행에 실패했습니다. 실행 기록의 오류를 확인하십시오.')
    finally:
        terminate(p);p.stdout.close();STATE['process']=None


def capture(args):
    check_cancel()
    p=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.PIPE,
                       text=True,start_new_session=True)
    STATE['process']=p
    try:
        while True:
            check_cancel()
            try:out,err=p.communicate(timeout=0.3);break
            except subprocess.TimeoutExpired:pass
        check_cancel()
        if p.returncode:raise RuntimeError(err[-3000:] or '외부 프로그램 실행 실패')
        return out
    finally:terminate(p);STATE['process']=None


def port_open():
    try:
        with socket.create_connection(('127.0.0.1',7860),timeout=0.3):return True
    except OSError:return False


CACHE=ModelCache(WORK_ROOT/'a1111-model-cache',cancel=STATE['cancel'],log=lambda msg:emit(msg))
CATALOG.update(CACHE.data['catalog'])
for item in PRESETS:
    CATALOG.setdefault(model_key(item),dict(item))
if 'selected' in CACHE.data:SELECTED.update(CACHE.data['selected'])
else:SELECTED.add(model_key(PRESETS[0]))
SELECTED.intersection_update(CATALOG)
if not port_open():
    CACHE.data['active']=[];CACHE.protected=set();CACHE.save()


def persist_catalog():
    CACHE.data['catalog']=CATALOG.copy()
    CACHE.data['selected']=sorted(SELECTED)
    CACHE.save()


def safe_stop():
    for key in ('tunnel_process','webui_process'):terminate(RUNTIME.get(key))
    for _ in range(15):
        if not port_open():break
        time.sleep(0.2)
    if port_open():raise RuntimeError('이 패널이 시작하지 않은 서버가 실행 중입니다. 기존 WebUI를 종료하거나 Kaggle 세션을 다시 시작하십시오.')
    if RUNTIME.get('AUTH_FILE'):Path(RUNTIME['AUTH_FILE']).unlink(missing_ok=True)
    CACHE.unlink_managed(WEBUI_ROOT)
    CACHE.protected=set()
    RUNTIME.pop('public_url',None);RUNTIME.pop('WEBUI_PASSWORD',None)
    connection.value=''


def secret(name,supplied=''):
    if supplied.strip():return supplied.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:return ''


def configure_downloads(tokens):
    RUNTIME.update(requests=requests,gdown=gdown,re=re,json=json,Path=Path,shutil=shutil,
                   urlparse=urlparse,parse_qs=parse_qs,unquote=unquote,
                   HEADERS={'User-Agent':'Kaggle-A1111-Manager/1.0'},
                   CIVITAI_TOKEN=tokens[0],HF_TOKEN=tokens[1])
    exec(compile(STAGES['다운로드 함수 준비'],'<download helpers>','exec'),RUNTIME)


def fetch_model(key):
    return CACHE.fetch(CATALOG[key],RUNTIME['civitai_file'],RUNTIME['request_headers'],requests.get,gdown.download)


def configure_cache():
    CACHE.limit=int(cache_limit.value*GIB);CACHE.reserve=int(disk_reserve.value*GIB)
    CACHE.data['limit_gib']=cache_limit.value;CACHE.data['reserve_gib']=disk_reserve.value
    CACHE.data['policy']=policy.value;CACHE.save()


def clone_extensions(names):
    root=WEBUI_ROOT/'extensions';root.mkdir(exist_ok=True)
    for name in names:
        check_cancel();repo=EXTENSIONS[name]
        folder=root/repo.rsplit('/',1)[-1].removesuffix('.git')
        if not (folder/'.git').is_dir():run(['git','clone','--depth','1',repo,folder],env=RUNTIME['probe_env'])


class UIWriter:
    def __init__(self):self.pending=''
    def write(self,text):
        self.pending+=text
        if '\n' in self.pending:
            parts=self.pending.split('\n');self.pending=parts.pop()
            for part in parts:
                if part:emit(part)
        return len(text)
    def flush(self):
        if self.pending:emit(self.pending);self.pending=''


def apply_models(keys,extensions,gpu_index,medvram_enabled):
    if not any(CATALOG[k]['category']=='Checkpoint' for k in keys):raise ValueError('사용할 Checkpoint를 하나 이상 선택하십시오.')
    items=[CATALOG[k] for k in keys]
    if all('Refiner' in item['label'] for item in items if item['category']=='Checkpoint'):
        raise ValueError('Refiner와 함께 사용할 기본 Checkpoint도 선택하십시오.')
    safe_stop();check_cancel()
    CACHE.protected=set(keys)
    # 시작 전에 캐시 한도에 맞춰 사용하지 않는 복사본을 정리합니다.
    if policy.value=='selected':CACHE.clean_unselected()
    CACHE.ensure_room()
    RUNTIME.update(GPU_INDEX=gpu_index,A1111_VERSION='v1.10.1',USE_MEDVRAM=medvram_enabled,
                   run=run,check_cancel=check_cancel,ACTIVE_LORA_DIR=WEBUI_ROOT/'models/Lora')
    for step,title in [(1,'환경 확인'),(2,'Python·WebUI 설치 및 GPU 확인')]:
        check_cancel();progress.value=step
        status.value=f'<b>{step}/5 · {html.escape(title)}</b><br>설치·다운로드 중에는 GPU 0%가 정상일 수 있습니다.'
        writer=UIWriter()
        with contextlib.redirect_stdout(writer),contextlib.redirect_stderr(writer):
            exec(compile(STAGES[title],'<'+title+'>','exec'),RUNTIME)
        writer.flush();check_cancel()
    names=list(extensions)
    if any(item['category']=='ControlNet' for item in items) and 'ControlNet' not in names:names.append('ControlNet')
    progress.value=3;status.value='<b>3/5 · 확장 설치·선택 모델 다운로드</b>'
    clone_extensions(names)
    if 'AnimateDiff' in names:
        motion=preset('AnimateDiff SD 1.5 motion','Motion','SD 1.5','conrevo/AnimateDiff-A1111','motion_module/mm_sd15_v3.safetensors')
        key=model_key(motion);CATALOG[key]=motion;keys=list(keys)+[key];CACHE.protected.add(key);items.append(motion)
    for index,key in enumerate(keys,1):
        check_cancel();emit(f'선택 모델 {index}/{len(keys)} · '+CATALOG[key]['label']);fetch_model(key)
    check_cancel();CACHE.ensure_room()
    RUNTIME['START_CHECKPOINT']=CACHE.activate(items,WEBUI_ROOT)
    progress.value=4;status.value='<b>4/5 · WebUI 시작 및 접속 준비</b>'
    writer=UIWriter()
    with contextlib.redirect_stdout(writer),contextlib.redirect_stderr(writer):
        exec(compile(STAGES['WebUI 시작 및 접속 준비'],'<WebUI launch>','exec'),RUNTIME)
    writer.flush();check_cancel()
    progress.value=5;progress.bar_style='success'
    show_connection()
    status.value='<b style="color:#15803d">5/5 · 접속 준비 완료</b> — 초록색 버튼을 누르십시오.'


FOLDER_CATEGORIES={'checkpoints':'Checkpoint','checkpoint':'Checkpoint','stable-diffusion':'Checkpoint',
                   'lora':'LoRA','loras':'LoRA','lycoris':'LoRA','controlnet':'ControlNet',
                   'embeddings':'Embedding','vae':'VAE'}


def folder_entries(files,category):
    entries=[];skipped=0
    for record in files:
        path=record['path'].replace('\\','/')
        parts=[p.lower() for p in path.split('/')[:-1]]
        if Path(path).suffix.lower() not in MODEL_SUFFIXES or 'outputs' in parts:
            skipped+=1;continue
        kind=category
        if kind=='Auto':
            kind=next((FOLDER_CATEGORIES[p] for p in parts if p in FOLDER_CATEGORIES),None)
            if kind is None:skipped+=1;continue
        entries.append(dict(label=path,category=kind,family='Drive',
                            url='https://drive.google.com/file/d/'+record['id']+'/view',
                            filename=model_filename(path)))
    return entries,skipped


def scan_drive(url,category):
    parsed=urlparse(url.strip())
    if parsed.scheme!='https' or parsed.hostname!='drive.google.com' or not re.search(r'/folders/[\w-]+',parsed.path):
        raise ValueError('Auto1111 또는 모델 하위 폴더의 Google Drive 공유 링크를 입력하십시오.')
    status.value='<b>Drive 폴더의 파일 목록을 읽는 중입니다. 모델은 다운로드하지 않습니다.</b>'
    program="""import gdown,json,sys
files=gdown.download_folder(url=sys.argv[1],skip_download=True,quiet=True,use_cookies=False,timeout=(15,30))
print(json.dumps([{'id':f.id,'path':f.path} for f in files],ensure_ascii=False))
"""
    raw=capture([sys.executable,'-c',program,url.strip()])
    entries,skipped=folder_entries(json.loads(raw),category)
    added=0
    for item in entries:
        key=model_key(item)
        if key not in CATALOG:added+=1
        CATALOG[key]=item
    persist_catalog()
    status.value=f'<b>목록 확인 완료:</b> 모델 {len(entries)}개, 새 등록 {added}개. 모델 파일은 다운로드하지 않았습니다. 기타 항목 {skipped}개는 제외했습니다.'
    emit('Drive 목록 등록 완료. 모델 관리 탭에서 사용할 모델을 선택하십시오.')


def validate_custom(category,url,filename):
    url=url.strip();filename=filename.strip();p=urlparse(url)
    if p.scheme!='https' or p.hostname not in {'civitai.com','www.civitai.com','huggingface.co','drive.google.com'}:
        raise ValueError('CivitAI·Hugging Face·Google Drive https 링크를 입력하십시오.')
    if p.username or p.password or any(k.lower() in {'token','api_key','apikey'} for k in parse_qs(p.query)):
        raise ValueError('토큰을 URL에서 제거하고 인증 입력란에 입력하십시오.')
    if p.hostname=='drive.google.com':
        if '/folders/' in p.path:raise ValueError('폴더 링크는 Drive 폴더 탭에서 등록하십시오.')
        if not filename:raise ValueError('Google Drive 개별 파일은 저장 파일명을 입력하십시오.')
    if p.hostname=='huggingface.co' and '/resolve/' not in p.path:raise ValueError('Hugging Face 파일 다운로드 링크(/resolve/)를 입력하십시오.')
    if filename:
        if '/' in filename or '\\' in filename:raise ValueError('저장 파일명에는 폴더 경로를 넣지 마십시오.')
        filename=model_filename(filename)
    return dict(label=filename or category+' 추가 모델',category=category,family='사용자 모델',url=url,filename=filename)


status=W.HTML('<b>모델을 선택하십시오.</b> Drive 폴더 목록은 파일을 다운로드하지 않고 읽습니다.')
progress=W.IntProgress(value=0,min=0,max=5,description='진행',layout=W.Layout(width='100%'))
connection=W.HTML('')
logs=W.Textarea(value='',disabled=True,layout=W.Layout(width='100%',height='230px'))
log_panel=W.Accordion(children=[logs]);log_panel.set_title(0,'실행 기록');log_panel.selected_index=None
stats=W.HTML('')
rows_box=W.VBox(layout=W.Layout(width='100%'))
page_prev=W.Button(description='이전',icon='chevron-left',layout=W.Layout(width='90px'))
page_next=W.Button(description='다음',icon='chevron-right',layout=W.Layout(width='90px'))
page_info=W.HTML('')
PAGE={'number':1}
category_filter=W.Dropdown(options=['전체','Checkpoint','LoRA','ControlNet','Embedding','VAE','Motion'],value='Checkpoint',description='목록 보기',style={'description_width':'initial'})
source_filter=W.Dropdown(options=['전체','Drive','다운로드됨','사용 선택됨'],value='전체',description='표시')
search=W.Text(placeholder='모델 이름 검색',layout=W.Layout(width='100%'))
apply_button=W.Button(description='선택 모델 적용·WebUI 실행',icon='play',button_style='success',layout=W.Layout(width='280px',height='46px'))
cancel_button=W.Button(description='작업 중단',icon='stop',button_style='danger',disabled=True,layout=W.Layout(width='150px',height='46px'))
show_button=W.Button(description='접속 버튼 표시',icon='external-link',layout=W.Layout(height='46px'))
stop_button=W.Button(description='WebUI 종료',icon='power-off')
zip_button=W.Button(description='이미지 ZIP',icon='download',button_style='info')
gpu_button=W.Button(description='GPU 확인',icon='microchip')
trim_button=W.Button(description='지금 캐시 정리',icon='trash')
drive_root_url=W.Text(value=CACHE.data.get('drive_folder_url',''),placeholder='Google Drive 모델 루트 폴더의 공유 링크',description='Drive 모델 폴더',style={'description_width':'initial'},layout=W.Layout(width='100%'))
initial_scan_button=W.Button(description='폴더 지정·목록 불러오기',icon='folder-open',button_style='info',layout=W.Layout(width='250px',height='42px'))
folder_kind=W.Dropdown(options=[('자동 분류 루트 폴더','Auto'),('Checkpoint 폴더','Checkpoint'),('LoRA/LyCORIS 폴더','LoRA'),('ControlNet 폴더','ControlNet'),('Embedding 폴더','Embedding'),('VAE 폴더','VAE')],value='Auto',description='폴더 종류',style={'description_width':'initial'},layout=W.Layout(width='100%'))
folder_url=W.Text(value=drive_root_url.value,placeholder='루트 폴더 또는 모델 하위 폴더의 공유 링크',description='폴더 링크',style={'description_width':'initial'},layout=W.Layout(width='100%'))
scan_button=W.Button(description='목록 불러오기',icon='folder-open',button_style='info')
file_kind=W.Dropdown(options=['Checkpoint','LoRA','ControlNet','Embedding','VAE'],description='모델 종류',style={'description_width':'initial'})
file_url=W.Text(placeholder='CivitAI·Hugging Face·Google Drive 개별 파일 링크',description='파일 링크',style={'description_width':'initial'},layout=W.Layout(width='100%'))
file_name=W.Text(placeholder='Google Drive 개별 파일은 필수. 예: model.safetensors',description='파일명',style={'description_width':'initial'},layout=W.Layout(width='100%'))
add_button=W.Button(description='모델 목록에 등록',icon='plus',button_style='info')
cache_limit=W.BoundedFloatText(value=float(CACHE.data.get('limit_gib',20)),min=1,max=1000,step=1,description='모델 한도 GiB',style={'description_width':'initial'})
disk_reserve=W.BoundedFloatText(value=float(CACHE.data.get('reserve_gib',3)),min=1,max=1000,step=1,description='최소 여유 GiB',style={'description_width':'initial'})
policy=W.Dropdown(options=[('한도 초과 시 오래 사용하지 않은 모델부터 정리','lru'),('선택·고정 모델만 보관','selected')],value=CACHE.data.get('policy','lru'),description='자동 정리',layout=W.Layout(width='100%'),style={'description_width':'initial'})
gpu=W.Dropdown(options=[('첫 번째 GPU',0),('두 번째 GPU',1)],value=0,description='GPU')
medvram=W.Checkbox(value=True,description='GPU 메모리 절약',indent=False)
civitai_token=W.Password(description='CivitAI 토큰',placeholder='선택 사항 / CIVITAI_API_TOKEN Secret 사용 가능',style={'description_width':'initial'},layout=W.Layout(width='100%'))
hf_token=W.Password(description='HF 토큰',placeholder='선택 사항 / HF_TOKEN Secret 사용 가능',style={'description_width':'initial'},layout=W.Layout(width='100%'))
ext_checks={name:W.Checkbox(value=False,description=name,indent=False) for name in EXTENSIONS}
zip_output=W.Output()
row_controls=[]
static_controls=[apply_button,show_button,stop_button,zip_button,gpu_button,trim_button,category_filter,source_filter,search,page_prev,page_next,drive_root_url,initial_scan_button,folder_kind,folder_url,scan_button,file_kind,file_url,file_name,add_button,cache_limit,disk_reserve,policy,gpu,medvram,civitai_token,hf_token,*ext_checks.values()]


def is_busy():return STATE.get('worker') is not None and STATE['worker'].is_alive()


def refresh_stats():
    free=shutil.disk_usage(WORK_ROOT).free/GIB
    order=['Checkpoint','LoRA','Embedding','ControlNet','VAE','Motion']
    counts={category:sum(1 for key in SELECTED if CATALOG.get(key,{}).get('category')==category) for category in order}
    selected_text=' · '.join(f'{category} {counts[category]}' for category in order if counts[category]) or '없음'
    stats.value=f'<b>현재 동시 선택:</b> {selected_text}<br><b>다운로드된 모델:</b> {CACHE.usage()/GIB:.2f} GiB / 한도 {cache_limit.value:g} GiB &nbsp; <b>Kaggle 전체 디스크 여유:</b> {free:.2f} GiB<br>등록 {len(CATALOG)}개 · 다음 실행 선택 {len(SELECTED)}개 · WebUI 연결 {len(CACHE.data["active"])}개 · 보관 고정 {len(CACHE.data["pins"])}개'


def on_select(key,value):
    if value:SELECTED.add(key)
    else:SELECTED.discard(key)
    persist_catalog();refresh_stats()


def on_pin(key,value):
    CACHE.pin(key,value);refresh_rows()


def matching_keys():
    found=[];needle=search.value.strip().lower()
    for key,item in CATALOG.items():
        if category_filter.value!='전체' and item['category']!=category_filter.value:continue
        if needle and needle not in item['label'].lower():continue
        cached=CACHE.cached_path(key)
        if source_filter.value=='Drive' and urlparse(item['url']).hostname!='drive.google.com':continue
        if source_filter.value=='다운로드됨' and not cached:continue
        if source_filter.value=='사용 선택됨' and key not in SELECTED:continue
        found.append(key)
    return found


def refresh_rows(*args,reset_page=False):
    global row_controls
    if reset_page:PAGE['number']=1
    built=[];row_controls=[]
    busy=is_busy();keys=matching_keys();size=25
    pages=max(1,(len(keys)+size-1)//size)
    PAGE['number']=min(max(1,PAGE['number']),pages)
    start=(PAGE['number']-1)*size;visible=keys[start:start+size]
    page_info.value=f'<b>모델 {len(keys)}개</b> · {start+1 if keys else 0}–{start+len(visible)} 표시 · {PAGE["number"]}/{pages}페이지'
    page_prev.disabled=busy or PAGE['number']<=1
    page_next.disabled=busy or PAGE['number']>=pages
    for key in visible:
        item=CATALOG[key];cached=CACHE.cached_path(key)
        active=key in CACHE.data['active'];pinned=key in CACHE.data['pins']
        use=W.Checkbox(value=key in SELECTED,description='사용',indent=False,disabled=busy,layout=W.Layout(width='68px'))
        use.observe(lambda change,k=key:on_select(k,change['new']),names='value')
        info='미다운로드 · 크기는 받을 때 확인'
        if cached:info=f'{cached.stat().st_size/GIB:.2f} GiB · 다운로드됨'
        if active:info+=' · WebUI 연결 중'
        name=W.HTML(f'<b>{html.escape(item["label"])}</b><br><small>{html.escape(item["category"])} · {html.escape(item["family"])} · {info}</small>',layout=W.Layout(flex='1 1 250px',min_width='180px'))
        download=W.Button(description='다운로드',icon='download',disabled=busy or bool(cached),layout=W.Layout(width='96px'))
        pin=W.ToggleButton(value=pinned,description='보관 고정',icon='thumb-tack',disabled=busy,button_style='warning' if pinned else '',layout=W.Layout(width='96px'))
        delete=W.Button(description='삭제',icon='trash',disabled=busy or not cached or active or pinned,layout=W.Layout(width='72px'))
        download.on_click(lambda button,k=key:start_download(k))
        pin.observe(lambda change,k=key:on_pin(k,change['new']),names='value')
        delete.on_click(lambda button,k=key:start_delete(k))
        row_controls.extend([use,download,pin,delete])
        built.append(W.HBox([use,name,download,pin,delete],layout=W.Layout(width='100%',align_items='center',flex_flow='row wrap',border='1px solid #e2e8f0',padding='6px')))
    rows_box.children=built or [W.HTML('표시할 모델이 없습니다. 필터를 바꾸거나 Drive 폴더 목록을 불러오십시오.')]
    refresh_stats()


def set_busy(value):
    for control in static_controls+row_controls:control.disabled=value
    cancel_button.disabled=not value
    if not value:
        page_prev.disabled=PAGE['number']<=1
        page_next.disabled=PAGE['number']>=max(1,(len(matching_keys())+24)//25)


def show_connection(_=None):
    p=RUNTIME.get('webui_process');t=RUNTIME.get('tunnel_process')
    if RUNTIME.get('public_url') and p is not None and p.poll() is None and t is not None and t.poll() is None:
        connection.value=connection_html(RUNTIME['public_url'],RUNTIME['WEBUI_USER'],RUNTIME['WEBUI_PASSWORD'])
    else:connection.value='<div style="padding:12px;background:#fff7ed">WebUI가 아직 준비되지 않았습니다. 사용할 모델을 선택하고 적용·실행을 누르십시오.</div>'


def start_task(label,fn,needs_tokens=False,is_apply=False):
    if is_busy():return
    STATE['cancel'].clear()
    configure_cache()
    tokens=[secret('CIVITAI_API_TOKEN',civitai_token.value),secret('HF_TOKEN',hf_token.value)] if needs_tokens else ['','']
    SECRET_VALUES[:]=[x for x in tokens if x]
    civitai_token.value='';hf_token.value=''
    log_panel.selected_index=0;logs.value='';progress.value=0;progress.bar_style='info'
    status.value=f'<b>{html.escape(label)}</b>'
    STATE['apply']=is_apply
    set_busy(True)
    def worker():
        try:
            configure_downloads(tokens)
            fn();check_cancel()
            if not is_apply:
                progress.bar_style='success'
                if label!='Drive 목록 읽기':status.value=f'<b style="color:#15803d">{html.escape(label)} 완료</b>'
        except BaseException as e:
            if is_apply:
                try:safe_stop()
                except Exception as cleanup_error:emit(str(cleanup_error))
            cancelled=STATE['cancel'].is_set() or isinstance(e,Cancelled)
            msg='작업을 중단했습니다. 완전히 받은 모델은 유지하고 불완전한 파일은 제거했습니다.' if cancelled else str(e)
            for token in SECRET_VALUES:msg=msg.replace(token,'[토큰 숨김]')
            emit(msg);progress.bar_style='warning' if cancelled else 'danger'
            status.value=f'<b>{"중단됨" if cancelled else "오류"}</b> · {html.escape(msg)}'
        finally:
            CACHE.protected=set(CACHE.data['active'])
            RUNTIME['CIVITAI_TOKEN']='';RUNTIME['HF_TOKEN']=''
            STATE['process']=None;STATE['worker']=None;SECRET_VALUES.clear()
            persist_catalog();set_busy(False);refresh_rows()
    thread=threading.Thread(target=worker,daemon=True)
    STATE['worker']=thread;thread.start()


def start_download(key):
    def task():
        CACHE.protected=set(CACHE.data['active'])|SELECTED|{key}
        fetch_model(key)
    start_task('모델 다운로드',task,needs_tokens=True)


def start_delete(key):
    def task():
        CACHE.protected=set(CACHE.data['active'])
        CACHE.delete(key,manual=True);SELECTED.discard(key)
    start_task('Kaggle 모델 복사본 삭제',task)


def change_page(delta):
    PAGE['number']+=delta;refresh_rows()


def on_apply(_):
    keys=[key for key in CATALOG if key in SELECTED]
    if not any(CATALOG[k]['category']=='Checkpoint' for k in keys):
        status.value='<b>Checkpoint를 하나 이상 사용 선택하십시오.</b>';return
    if any('Refiner' not in CATALOG[k]['label'] for k in keys if CATALOG[k]['category']=='Checkpoint'):
        keys.sort(key=lambda k: CATALOG[k]['category']=='Checkpoint' and 'Refiner' in CATALOG[k]['label'])
    names=[name for name,check in ext_checks.items() if check.value]
    index=gpu.value;memory=medvram.value
    start_task('선택 모델 적용',lambda:apply_models(keys,names,index,memory),needs_tokens=True,is_apply=True)


def on_cancel(_):
    STATE['cancel'].set()
    terminate(STATE.get('process'),wait=False)
    if STATE.get('apply'):
        for key in ('tunnel_process','webui_process'):terminate(RUNTIME.get(key),wait=False)
    status.value='<b>중단 요청을 받았습니다.</b> 현재 파일·프로세스를 정리하고 있습니다. 네트워크 응답 대기 중이면 시간이 걸릴 수 있습니다.'


def on_scan(_):
    url=folder_url.value;kind=folder_kind.value
    def task():
        scan_drive(url,kind)
        if kind=='Auto':
            drive_root_url.value=url.strip();CACHE.data['drive_folder_url']=url.strip();CACHE.save()
        category_filter.value='전체';source_filter.value='Drive';search.value=''
    start_task('Drive 목록 읽기',task)


def on_initial_scan(_):
    url=drive_root_url.value.strip()
    def task():
        scan_drive(url,'Auto')
        CACHE.data['drive_folder_url']=url;CACHE.save()
        folder_url.value=url;folder_kind.value='Auto'
        category_filter.value='전체';source_filter.value='Drive';search.value=''
    start_task('Drive 목록 읽기',task)


def on_add(_):
    try:
        item=validate_custom(file_kind.value,file_url.value,file_name.value)
        CATALOG[model_key(item)]=item;persist_catalog()
        category_filter.value=item['category'];source_filter.value='전체';search.value=''
        file_url.value='';file_name.value='';refresh_rows()
        status.value='<b>모델 목록에 등록했습니다.</b> 사용 체크 또는 다운로드 버튼을 누르십시오.'
    except Exception as e:status.value='<b>입력 확인:</b> '+html.escape(str(e))


def on_trim(_):
    def task():
        CACHE.protected=set(CACHE.data['active'])|SELECTED
        if policy.value=='selected':CACHE.clean_unselected()
        CACHE.ensure_room()
    start_task('캐시 자동 정리',task)


def on_gpu(_):
    def task():
        emit(capture(['nvidia-smi']))
        path=WORK_ROOT/'webui.log'
        if path.exists():emit(path.read_text(errors='ignore')[-6000:])
    start_task('GPU·실행 로그 확인',task)


def on_zip(_):
    def task():
        generated=WORK_ROOT/'downloads/generated'
        files=[p for p in generated.rglob('*') if p.is_file()]
        if not files:raise RuntimeError('생성된 이미지가 없습니다.')
        output=generated.parent/'stable_diffusion_generated_images.zip'
        partial=output.with_suffix('.zip.part')
        try:
            with zipfile.ZipFile(partial,'w',compression=zipfile.ZIP_STORED) as z:
                for file in files:check_cancel();z.write(file,file.relative_to(generated))
            check_cancel();partial.replace(output)
        except BaseException:partial.unlink(missing_ok=True);raise
        os.chdir(WORK_ROOT)
        with zip_output:
            zip_output.clear_output(wait=True)
            display(FileLink(str(output.relative_to(WORK_ROOT))))
    start_task('이미지 ZIP 만들기',task)

apply_button.on_click(on_apply);cancel_button.on_click(on_cancel);show_button.on_click(show_connection)
initial_scan_button.on_click(on_initial_scan);scan_button.on_click(on_scan);add_button.on_click(on_add);trim_button.on_click(on_trim)
stop_button.on_click(lambda _:start_task('WebUI 종료',safe_stop));gpu_button.on_click(on_gpu);zip_button.on_click(on_zip)
page_prev.on_click(lambda _:change_page(-1));page_next.on_click(lambda _:change_page(1))
for widget in (category_filter,source_filter,search):widget.observe(lambda change:refresh_rows(reset_page=True),names='value')
model_tab=W.VBox([W.HTML('<b>모델 검색 및 선택</b><br>한 페이지에 25개씩 표시합니다. 다른 페이지에서 체크한 사용 선택도 유지됩니다.'),W.HBox([category_filter,source_filter]),search,W.HBox([page_prev,page_next,page_info],layout=W.Layout(flex_flow='row wrap',align_items='center',gap='8px')),rows_box])
drive_tab=W.VBox([W.HTML('<b>Google Drive 폴더 상세 지정</b><br>맨 위에서 지정한 루트 폴더를 바꾸거나 Checkpoint·LoRA 등 특정 하위 폴더만 별도로 불러올 때 사용하십시오.<br>폴더를 「링크가 있는 모든 사용자 → 뷰어」로 공유하십시오. 목록만 읽으며 모델 파일은 받지 않습니다.'),folder_kind,folder_url,scan_button])
file_tab=W.VBox([W.HTML('폴더 대신 개별 모델을 등록할 때 사용하십시오. Google Drive는 파일명도 입력합니다.'),file_kind,file_url,file_name,add_button])
options_tab=W.VBox([cache_limit,disk_reserve,policy,trim_button,W.HTML('모델 한도에는 이 관리 기능으로 받은 모델만 포함됩니다. 설치 파일·기존 파일·이미지는 제외됩니다.'),gpu,medvram,civitai_token,hf_token,W.HTML('<b>확장 프로그램</b> · ControlNet 모델 선택 시 확장은 자동 설치됩니다.'),*ext_checks.values()])
tabs=W.Tab(children=[model_tab,drive_tab,file_tab,options_tab])
for i,title in enumerate(['모델 관리','Drive 폴더','파일 링크','용량·확장·인증']):tabs.set_title(i,title)
drive_setup=W.VBox([W.HTML('<h3 style="margin:4px 0">1. Google Drive 모델 폴더 지정</h3><p style="margin:4px 0 8px">사용할 루트 폴더는 이름과 위치가 자유롭습니다. 공유 폴더 링크를 입력하면 Checkpoint·LoRA·LyCORIS·ControlNet·Embedding·VAE를 자동 분류합니다. Google Drive를 사용하지 않으면 비워 두십시오.</p>'),drive_root_url,initial_scan_button],layout=W.Layout(padding='10px',border='1px solid #38bdf8'))
panel=W.VBox([W.HTML('<h2>AUTOMATIC1111 · 학생 개인 실행</h2><p>각자의 Kaggle 세션에서 모델 선택 → WebUI 실행 → 이미지 다운로드 → 세션 종료</p>'),drive_setup,status,progress,connection,W.HBox([apply_button,cancel_button,show_button],layout=W.Layout(flex_flow='row wrap',gap='6px')),stats,tabs,W.HBox([stop_button,gpu_button,zip_button]),zip_output,log_panel],layout=W.Layout(width='100%',max_width='1000px',padding='12px',border='1px solid #cbd5e1'))
persist_catalog();refresh_rows();display(panel)
if RUNTIME.get('public_url'):show_connection()


## 학생용 사용 안내

- **개인 계정 실행**: 학생마다 자신의 Kaggle 계정에서 노트북을 복사하고 실행하십시오. 다른 학생의 Kaggle 계정이나 실행 주소를 함께 사용하지 마십시오.
- **임시 로그인**: WebUI 사용자명 `a1111`과 임시 비밀번호는 Kaggle 계정 정보가 아닙니다. 비밀번호는 실행할 때마다 새로 생성되며 화면에서는 가려집니다. **비밀번호 복사** 버튼을 사용하십시오.

- **Google Drive 폴더 지정**: 폴더를 `링크가 있는 모든 사용자 → 뷰어`로 공유하고 폴더 링크를 맨 위에 입력한 뒤 **폴더 지정·목록 불러오기**를 누르십시오. 모델 목록은 검색하고 25개씩 넘겨 볼 수 있습니다.
- **자동 분류 폴더**: `checkpoints`, `Lora`, `LyCORIS`, `ControlNet`, `embeddings`, `VAE` 폴더의 모델을 종류별로 분류합니다. **목록 불러오기** 단계에서는 모델 파일을 다운로드하지 않습니다.
- **모델 다운로드**: **사용**으로 선택한 모델은 WebUI 실행 시 다운로드합니다. 이미 받은 모델은 다시 받지 않고 재사용합니다.
- **개별 링크**: CivitAI 모델 주소, Hugging Face `/resolve/` 다운로드 주소, Google Drive 개별 파일 공유 주소를 등록할 수 있습니다. Google Drive 개별 파일은 저장할 파일명도 입력하십시오.
- **인증 토큰**: 필요한 경우 Kaggle Secrets에 `CIVITAI_API_TOKEN` 또는 `HF_TOKEN`을 저장하거나 실행 패널의 토큰 입력란을 사용하십시오.
- **모델 호환성**: Checkpoint의 계열에 맞는 SD 1.5 또는 SDXL용 LoRA·ControlNet·VAE를 선택하십시오.
- **모델 교체**: **사용** 선택을 바꾼 뒤 **선택 모델 적용·WebUI 실행**을 다시 누르십시오. WebUI가 새 선택으로 재시작됩니다.
- **용량 관리**: 모델 캐시 한도와 최소 여유 공간은 **실행 설정**에서 조정할 수 있습니다. 선택한 모델, 실행 중인 모델, 보관 고정 모델은 자동 정리에서 보호됩니다.
- **GPU 사용 시점**: 설치와 다운로드 중에는 GPU 사용률이 낮을 수 있습니다. WebUI에서 이미지를 생성할 때 GPU를 사용합니다.
- **중단**: 설치·다운로드 중에는 **작업 중단**, 실행 후에는 **WebUI 종료**를 사용하십시오. Kaggle 세션 자체를 끝내려면 오른쪽 위 세션 메뉴에서 **Stop Session**을 선택하십시오.
- **생성 이미지**: 결과는 `/kaggle/working/downloads/generated`에 저장됩니다. 세션을 종료하기 전에 **이미지 ZIP**으로 내려받으십시오.
- **접속 정보**: WebUI 준비가 끝나면 실행 패널 상단에 개인 접속 주소, 사용자명 `a1111`, 임시 비밀번호가 표시됩니다. 비밀번호는 기본적으로 가려집니다. 로그인 화면에 브라우저가 저장한 값이 자동 입력되면 지우고 이 정보를 입력하십시오.

사용 안내: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks) · [AUTOMATIC1111](https://github.com/AUTOMATIC1111/stable-diffusion-webui) · [gdown](https://github.com/wkentaro/gdown) · [ControlNet](https://github.com/Mikubill/sd-webui-controlnet)
